## 5. Distribución y umbral de anomalía por modelo

Cada modelo produce su propio `decision_score_` con escala y distribución distintas.
Usamos **percentil 95 (P95)** como umbral de anomalía en ambos — el 5% de nodos
con mayor score se etiquetan como anómalos. Este umbral es consistente con el parámetro
`contamination=0.05` pasado a los dos detectores.

La comparativa directa de los scores brutos no tiene sentido porque sus escalas son
incomparables (DOMINANT: RMSE sobre adyacencia densa; GAD-NR: combinación ponderada
de tres pérdidas). Lo que sí es comparable es el **ranking** que cada modelo asigna
a los nodos y las **poblaciones** que cada uno etiqueta como anómalas.

In [ ]:
# Scores finales de cada modelo
dom_score   = dominant.decision_score_.cpu().numpy()   # score combinado DOMINANT
gadnr_score = gadnr.decision_score_.cpu().numpy()      # score combinado GAD-NR

# Percentiles P95 de cada modelo
p95_dom   = np.percentile(dom_score,   95)
p95_gadnr = np.percentile(gadnr_score, 95)

# Etiquetas binarias por modelo (1 = anómalo)
labels_dom   = (dom_score   >= p95_dom).astype(int)
labels_gadnr = (gadnr_score >= p95_gadnr).astype(int)

print('=== DOMINANT ===')
print(f'  Score  — min={dom_score.min():.4f}, max={dom_score.max():.4f}, mean={dom_score.mean():.4f}')
print(f'  P95    = {p95_dom:.4f}')
print(f'  Anomalías detectadas: {labels_dom.sum()} ({labels_dom.mean():.1%})')
print()
print('=== GAD-NR ===')
print(f'  Score  — min={gadnr_score.min():.4f}, max={gadnr_score.max():.4f}, mean={gadnr_score.mean():.4f}')
print(f'  P95    = {p95_gadnr:.4f}')
print(f'  Anomalías detectadas: {labels_gadnr.sum()} ({labels_gadnr.mean():.1%})')

In [ ]:
# Distribuciones de score por modelo
fig, axes = plt.subplots(1, 2, figsize=FS['r1c2'])

for ax, scores, p95, label, color in zip(
    axes,
    [dom_score, gadnr_score],
    [p95_dom,   p95_gadnr],
    ['DOMINANT', 'GAD-NR'],
    [PALETTE['M'], PALETTE['MG']]
):
    ax.hist(scores, bins=80, color=color, edgecolor='white', alpha=0.85)
    ax.axvline(p95, color=PALETTE['anomaly'], linestyle='--', linewidth=1.0,
               label=f'P95 = {p95:.4f}')
    ax.set_xlabel('Score de anomalía', fontsize=FONT['axis_label'])
    ax.set_ylabel('Nº de nodos', fontsize=FONT['axis_label'])
    ax.set_title(label, fontsize=FONT['title'])
    ax.legend(fontsize=FONT['legend'], frameon=False)
    polish(ax)

plt.suptitle('Distribución del score de anomalía — Grafo M', fontsize=FONT['title'], y=1.02)
plt.tight_layout()
save_figure(fig, '05_score_distributions_M.png')
plt.show()

In [ ]:
# Añadimos scores y etiquetas al DataFrame base
scores_df = base_df.copy()
scores_df['dom_score']     = dom_score
scores_df['gadnr_score']   = gadnr_score
scores_df['pct_dom']       = rankdata(dom_score)   / len(dom_score)   * 100
scores_df['pct_gadnr']     = rankdata(gadnr_score) / len(gadnr_score) * 100
scores_df['label_dom']     = labels_dom
scores_df['label_gadnr']   = labels_gadnr

print('Scores añadidos al DataFrame:')
print(scores_df[['name', 'degree', 'dom_score', 'gadnr_score',
                 'pct_dom', 'pct_gadnr', 'label_dom', 'label_gadnr']].head(10))

## 6. Spotlight — candidatos del EDA

Revisamos los candidatos identificados en el EDA con la perspectiva de ambos modelos.
Para cada candidato mostramos el percentil en cada score y si supera el umbral P95.

Según los papers:
- **DOMINANT** detecta anomalías **estructurales** (conectividad inusual) y **contextuales**
  (features distintas a la mayoría), pero **no** joint-type.
- **GAD-NR** detecta los tres tipos: estructural, contextual y **joint-type** (nodos con
  muchas conexiones hacia nodos de features distintas) — el tipo que DOMINANT no captura.

In [ ]:
def spotlight(name_fragment, df, label=None):
    """Imprime el perfil de anomalía de un candidato del EDA."""
    matches = df[df['name'].str.contains(name_fragment, case=False, na=False)]
    if len(matches) == 0:
        print(f'  [no encontrado] {name_fragment}\n')
        return

    row = matches.iloc[0]
    n   = len(df)

    def rank(col):
        return int((df[col] > row[col]).sum()) + 1

    flag_dom   = '⚠ ANOMALÍA' if row['label_dom']   else 'Normal'
    flag_gadnr = '⚠ ANOMALÍA' if row['label_gadnr'] else 'Normal'

    tag = label or str(row['name'])
    print(f'  ► {tag}')
    print(f'    Ciudad: {row["city"]} ({row["state"]}) | Grado: {row["degree"]}')
    print(f'    DOMINANT : #{rank("dom_score"):5d}/{n}  ({row["pct_dom"]:.1f}p)  → {flag_dom}')
    print(f'    GAD-NR   : #{rank("gadnr_score"):5d}/{n}  ({row["pct_gadnr"]:.1f}p)  → {flag_gadnr}')
    print()


print('=' * 70)
print('CANDIDATOS DEL EDA — DOMINANT vs GAD-NR')
print('=' * 70 + '\n')

candidatos = [
    ('Pablo',         'Pablo (alta betweenness, degree moderada)'),
    ('Shalini',       'Shalini (mayor betweenness de la red)'),
    ('Jim H',         'Jim H (super-conector + puente inter-comunidad)'),
    ('Matt Kenigson', 'Matt Kenigson (super-conector)'),
    ('GEEK',          'GEEK by AKEIN Engineering (entidad no personal)'),
]

for frag, label in candidatos:
    spotlight(frag, scores_df, label=label)

In [ ]:
# Tabla resumen ejecutivo
rows = []
for frag, label in candidatos:
    matches = scores_df[scores_df['name'].str.contains(frag, case=False, na=False)]
    if len(matches) == 0:
        continue
    row = matches.iloc[0]
    n   = len(scores_df)

    def rank(col):
        return int((scores_df[col] > row[col]).sum()) + 1

    rows.append({
        'Candidato':  label.split('(')[0].strip(),
        'Grado':      int(row['degree']),
        'DOMINANT':   f'#{rank("dom_score")} ({row["pct_dom"]:.0f}p) {"⚠" if row["label_dom"] else ""}',
        'GAD-NR':     f'#{rank("gadnr_score")} ({row["pct_gadnr"]:.0f}p) {"⚠" if row["label_gadnr"] else ""}',
    })

print('=== Resumen ejecutivo — candidatos del EDA ===')
print(pd.DataFrame(rows).to_string(index=False))

## 7. Comparativa DOMINANT vs GAD-NR

### 7.1 Correlación de rankings

La correlación de Spearman entre los rankings de los dos modelos indica en qué medida
coinciden en su valoración de la anomalidad de los nodos. Una correlación alta indicaría
que ambos modelos detectan esencialmente las mismas anomalías. Una correlación baja
indicaría que GAD-NR aporta información genuinamente nueva al capturar el vecindario
completo — especialmente las anomalías joint-type que DOMINANT no puede detectar.

### 7.2 Solapamiento del top-K

El solapamiento del top-K complementa la correlación: mide qué fracción de los K nodos
más anómalos según un modelo también aparece en el top-K del otro.

### 7.3 Nodos exclusivos de cada modelo

Los nodos que aparecen en el top-K de un modelo pero no del otro son los más interesantes
para el análisis: revelan qué tipo de anomalía captura exclusivamente cada arquitectura
según la taxonomía de los papers originales.

In [ ]:
# Correlación de Spearman entre los dos scores
rho, pval = spearmanr(dom_score, gadnr_score)
print(f'Correlación Spearman DOMINANT vs GAD-NR: ρ = {rho:.4f} (p = {pval:.2e})')
print()

# Solapamiento del top-K para distintos valores de K
print('=== Solapamiento del top-K ===')
for K in [50, 100, 200, 500]:
    top_dom   = set(np.argsort(-dom_score)[:K])
    top_gadnr = set(np.argsort(-gadnr_score)[:K])
    overlap   = len(top_dom & top_gadnr)
    print(f'  Top-{K:4d}: {overlap:3d} nodos comunes ({overlap/K:.0%} solapamiento)')

In [ ]:
# Scatter DOMINANT vs GAD-NR — un punto por nodo
# Coloreamos por categoría de acuerdo (ambos detectan / solo uno / ninguno)
ambos    = (scores_df['label_dom'] == 1) & (scores_df['label_gadnr'] == 1)
solo_dom = (scores_df['label_dom'] == 1) & (scores_df['label_gadnr'] == 0)
solo_gad = (scores_df['label_dom'] == 0) & (scores_df['label_gadnr'] == 1)
ninguno  = (scores_df['label_dom'] == 0) & (scores_df['label_gadnr'] == 0)

print('=== Acuerdo entre modelos (P95) ===')
print(f'  Ambos detectan    : {ambos.sum():5d} nodos ({ambos.mean():.1%})')
print(f'  Solo DOMINANT     : {solo_dom.sum():5d} nodos ({solo_dom.mean():.1%})')
print(f'  Solo GAD-NR       : {solo_gad.sum():5d} nodos ({solo_gad.mean():.1%})')
print(f'  Ninguno           : {ninguno.sum():5d} nodos ({ninguno.mean():.1%})')

fig, ax = plt.subplots(figsize=FS['wide'])

grupos = [
    (ninguno,  PALETTE['neutral'],  0.15, 4,  'Normal (ninguno)'),
    (solo_dom, PALETTE['M'],        0.8,  40, f'Solo DOMINANT ({solo_dom.sum()})'),
    (solo_gad, PALETTE['MG'],       0.8,  40, f'Solo GAD-NR ({solo_gad.sum()})'),
    (ambos,    PALETTE['anomaly'],  0.9,  60, f'Ambos ({ambos.sum()})'),
]

for mask, color, alpha, size, label in grupos:
    ax.scatter(
        scores_df.loc[mask, 'pct_dom'],
        scores_df.loc[mask, 'pct_gadnr'],
        s=size, alpha=alpha, color=color, label=label,
        zorder=3 if label != 'Normal (ninguno)' else 1
    )

# Líneas de umbral P95
ax.axvline(95, color=PALETTE['M'],  linestyle='--', linewidth=0.8, alpha=0.7)
ax.axhline(95, color=PALETTE['MG'], linestyle='--', linewidth=0.8, alpha=0.7)

# Etiquetamos candidatos del EDA
for frag, label in candidatos:
    matches = scores_df[scores_df['name'].str.contains(frag, case=False, na=False)]
    if len(matches) == 0:
        continue
    row = matches.iloc[0]
    ax.annotate(
        str(row['name'])[:15],
        (row['pct_dom'], row['pct_gadnr']),
        fontsize=FONT['annotation'],
        xytext=(5, 3), textcoords='offset points',
        arrowprops=dict(arrowstyle='->', color='black', lw=0.5)
    )

ax.set_xlabel('DOMINANT — Percentil del score', fontsize=FONT['axis_label'])
ax.set_ylabel('GAD-NR — Percentil del score',   fontsize=FONT['axis_label'])
ax.set_title('Comparativa de rankings: DOMINANT vs GAD-NR (Grafo M)',
             fontsize=FONT['title'])
ax.legend(title='Detección (P95)', fontsize=FONT['legend'], frameon=False)
polish(ax, grid=False)
plt.tight_layout()
save_figure(fig, '05_dominant_vs_gadnr_scatter_M.png')
plt.show()

In [ ]:
# Nodos exclusivos de cada modelo — top-100
K = 100
top_dom_idx   = set(np.argsort(-dom_score)[:K])
top_gadnr_idx = set(np.argsort(-gadnr_score)[:K])

only_dom   = top_dom_idx   - top_gadnr_idx
only_gadnr = top_gadnr_idx - top_dom_idx

cols_show = ['name', 'city', 'state', 'degree', 'pct_dom', 'pct_gadnr']

print(f'=== Top-{K} exclusivos de DOMINANT (no en top-{K} de GAD-NR) ===')
print('→ Probables anomalías estructurales o contextuales puras')
excl_dom = scores_df.iloc[sorted(only_dom)].nlargest(10, 'dom_score')[cols_show]
print(excl_dom.to_string(index=False))

print(f'\n=== Top-{K} exclusivos de GAD-NR (no en top-{K} de DOMINANT) ===')
print('→ Probables anomalías joint-type: conectados a nodos de features distintas')
excl_gadnr = scores_df.iloc[sorted(only_gadnr)].nlargest(10, 'gadnr_score')[cols_show]
print(excl_gadnr.to_string(index=False))

In [ ]:
# Perfil de grado de los nodos exclusivos de cada modelo
# ¿Los nodos que solo detecta GAD-NR tienen grados distintos a los de DOMINANT?
fig, ax = plt.subplots(figsize=FS['single'])

deg_only_dom   = scores_df.iloc[sorted(only_dom)]['degree']
deg_only_gadnr = scores_df.iloc[sorted(only_gadnr)]['degree']

ax.hist(deg_only_dom,   bins=30, alpha=0.7, color=PALETTE['M'],
        label=f'Solo DOMINANT (n={len(only_dom)})',   edgecolor='white')
ax.hist(deg_only_gadnr, bins=30, alpha=0.7, color=PALETTE['MG'],
        label=f'Solo GAD-NR (n={len(only_gadnr)})', edgecolor='white')

ax.set_xlabel('Grado del nodo', fontsize=FONT['axis_label'])
ax.set_ylabel('Nº de nodos',   fontsize=FONT['axis_label'])
ax.set_title('Distribución de grado — nodos exclusivos de cada modelo',
             fontsize=FONT['title'])
ax.legend(fontsize=FONT['legend'], frameon=False)
polish(ax)
plt.tight_layout()
save_figure(fig, '05_degree_exclusive_nodes_M.png')
plt.show()

print(f'Grado medio — solo DOMINANT : {deg_only_dom.mean():.1f}')
print(f'Grado medio — solo GAD-NR   : {deg_only_gadnr.mean():.1f}')
print(f'Grado medio — grafo completo: {scores_df["degree"].mean():.1f}')

## 8. Guardado de resultados

In [ ]:
cols_export = [
    'idx', 'member_id', 'name', 'city', 'state', 'degree',
    # Scores brutos
    'dom_struct', 'dom_attr', 'dom_score',
    'gadnr_h', 'gadnr_deg', 'gadnr_feat', 'gadnr_score',
    # Percentiles
    'pct_dom', 'pct_gadnr',
    # Etiquetas binarias
    'label_dom', 'label_gadnr',
]

# Añadimos sub-scores de DOMINANT y GAD-NR si están disponibles
scores_df['dom_struct']  = dom_struct
scores_df['dom_attr']    = dom_attr
scores_df['gadnr_h']     = gadnr_h
scores_df['gadnr_deg']   = gadnr_deg
scores_df['gadnr_feat']  = gadnr_feat

scores_export = scores_df[[c for c in cols_export if c in scores_df.columns]]
scores_export.to_csv(os.path.join(RESULTS_PATH, 'scores_M.csv'), index=False)

print(f'Guardado : {RESULTS_PATH}scores_M.csv')
print(f'Shape    : {scores_export.shape}')
print(f'Columnas : {list(scores_export.columns)}')

## Resumen del notebook

En este notebook hemos:

1. **Entrenado DOMINANT** (PyGOD 1.1.0) sobre el grafo M — línea base establecida
   en la literatura para detección de anomalías en grafos atribuidos.

2. **Entrenado GAD-NR** (PyGOD 1.1.0) sobre el mismo grafo — estado del arte que
   extiende el paradigma GAE con reconstrucción de vecindario completo.

3. **Comparado los scores** de ambos modelos mediante correlación de Spearman,
   solapamiento del top-K y análisis de nodos exclusivos de cada detector.

4. **Validado los candidatos del EDA** — Pablo, Shalini, Jim H, Matt Kenigson y
   GEEK by AKEIN Engineering — contrastando el diagnóstico de cada modelo.

5. **Caracterizado los nodos exclusivos** de cada modelo: los que solo detecta
   DOMINANT son probables anomalías estructurales o contextuales puras; los que
   solo detecta GAD-NR son probables anomalías joint-type — nodos con patrones
   de conectividad hacia vecinos de features distintas, el tipo que DOMINANT
   no puede detectar según el paper original (Roy et al., 2024).

Los resultados quedan en `data/results/scores_M.csv`.